# Inter-area connectivity — Blind vs Sighted

Analysis notebook for the connectivity results (paradigms **P1** and **P2**).

This notebook only *loads* the pre-built dataset and calls the `connectivity`
package. If `data/connectivity_data.csv` does not exist yet, build it once:

```bash
python scripts/build_dataset.py --sighted-dir … --blind-dir … --tally-baseline …
```

The 12 areas are grouped into four functional systems: **Visual** (V1, TO, AT),
**Motor** (PFL, PML, M1L), **Auditory** (A1, AB, PB), **Articulatory** (PFI, PMI, M1I).

## Setup

In [ ]:
# Make the `connectivity` package importable from the notebooks/ folder.
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9.5, 8.5)

from connectivity import load_dataset, area_connectivity, undirected_bidir, plotting
from connectivity.stats import (
    system_pair_blind_vs_sighted_by_net,
    within_group_change_test,
)

In [ ]:
# Load the processed dataset (one row per subject / presentation / area-pair).
data = load_dataset()

# Directed mean connectivity, and an undirected (symmetrised) version for figures.
area_conn = area_connectivity(data)
area_undirected = undirected_bidir(area_conn)

data.head()

## 1. Connectivity matrices

Absolute area-by-area connectivity for one condition at one presentation.

In [ ]:
plotting.plot_connectivity_triangle(area_undirected, "P1", "Sighted", presentation=0,    vmax=1000)
plotting.plot_connectivity_triangle(area_undirected, "P1", "Sighted", presentation=5000, vmax=1000)
plotting.plot_connectivity_triangle(area_undirected, "P1", "Blind",   presentation=5000, vmax=1000)

## 2. Blind − Sighted differences

Difference in connectivity between conditions at a fixed presentation.

In [ ]:
plotting.plot_diff_blind_sighted_triangle(area_undirected, "P1", presentation=0)
plotting.plot_diff_blind_sighted_triangle(area_undirected, "P1", presentation=5000)
plotting.plot_diff_blind_sighted_triangle(area_undirected, "P2", presentation=50)

## 3. Change over training

Connectivity change within a condition, `P{v2} − P{v1}`.
P1 uses 0 → 5000; P2 uses 0 → 50.

In [ ]:
plotting.plot_change_heatmap(area_undirected, "P1", "Sighted", v1=0, v2=5000, vmax=2000, triangle="lower")
plotting.plot_change_heatmap(area_undirected, "P1", "Blind",   v1=0, v2=5000, vmax=2000, triangle="lower")

plotting.plot_change_heatmap(area_undirected, "P2", "Sighted", v1=0, v2=50, vmax=60, triangle="lower")
plotting.plot_change_heatmap(area_undirected, "P2", "Blind",   v1=0, v2=50, vmax=60, triangle="lower")

## 4. Condition contrast of change

`(Blind change) − (Sighted change)` over the same presentation window.

In [ ]:
plotting.plot_change_contrast_heatmap(area_undirected, "P2", cond_a="Blind", cond_b="Sighted",
                                      v1=0, v2=50, triangle="lower")

## 5. Network-level statistics

For every functional-system pair, aggregate connectivity per network and compare
Blind vs Sighted across networks (paired when every network has both conditions,
otherwise Student's t-test), Bonferroni-corrected.

In [ ]:
result_p1 = system_pair_blind_vs_sighted_by_net(data, "P1", presentation=5000, include_diagonal=True)

In [ ]:
result_p2 = system_pair_blind_vs_sighted_by_net(data, "P2", presentation=50, include_diagonal=True)

### Within-system change test

Does the training change in within-system connectivity differ between conditions?

In [ ]:
_ = within_group_change_test(data, "P2", p_start=0, p_end=50, areas=("V1", "TO", "AT"))